# Operator Overloading in Python — Explained Like You Are 5

## Goal

**Operator overloading** teaches symbols such as `+`, `-`, `*`, `/`, and `==` how to work with objects we create.

Imagine that `+` is a helper robot:

- Give it two numbers and it adds them.
- Give it two strings and it joins them.
- Give it two vectors and we can teach it to add their coordinates.

> **Big idea:** The operator stays the same, but its behavior depends on the objects around it.

## Operator overloading uses magic methods

When Python sees an operator, it translates it into a magic-method call.

```python
left + right       # left.__add__(right)
left == right      # left.__eq__(right)
-value             # value.__neg__()
```

We usually write the friendly operator syntax, not the direct magic-method call.

## Quick operator map

| Operator | Magic method | Meaning |
|---|---|---|
| `+` | `__add__` | Addition |
| `-` | `__sub__` | Subtraction |
| `*` | `__mul__` | Multiplication |
| `/` | `__truediv__` | True division |
| `//` | `__floordiv__` | Floor division |
| `%` | `__mod__` | Remainder |
| `**` | `__pow__` | Power |
| `==` | `__eq__` | Equality |
| `<` | `__lt__` | Less than |
| `<=` | `__le__` | Less than or equal |
| `>` | `__gt__` | Greater than |
| `>=` | `__ge__` | Greater than or equal |
| `-obj` | `__neg__` | Unary negative |
| `obj += value` | `__iadd__` | In-place addition |

## 1. Python already overloads operators

Python gives `+` different jobs for numbers, strings, and lists. Custom classes can join this system.

In [1]:
print("Numbers:", 2 + 3)
print("Strings:", "Hello " + "World")
print("Lists:", [1, 2] + [3, 4])

Numbers: 5
Strings: Hello World
Lists: [1, 2, 3, 4]


## 2. Vector mathematics — original example

A vector has an `x` and `y` value. The class teaches Python these rules:

- `v1 + v2`: add matching coordinates.
- `v1 - v2`: subtract matching coordinates.
- `v1 * 3`: multiply each coordinate by 3.
- `v1 == v2`: compare both coordinates.
- `repr(v1)`: display the result clearly.

Every arithmetic method returns a **new Vector** instead of changing the original vectors.

In [2]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, other):
        return Vector(self.x * other, self.y * other)

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"


v1 = Vector(2, 3)
v2 = Vector(4, 5)

print(v1 + v2)
print(v1 - v2)
print(v1 * 3)
print(v1 == v2)

Vector(6, 8)
Vector(-2, -2)
Vector(6, 9)
False


### Confirming that operations create new objects

`v1 + v2` produces a result without changing either input. This predictable style is often easier to understand.

In [3]:
result = v1 + v2

print("v1 is still:", v1)
print("v2 is still:", v2)
print("New result:", result)

v1 is still: Vector(2, 3)
v2 is still: Vector(4, 5)
New result: Vector(6, 8)


## 3. Complex numbers — original example

For `(a + bi)` and `(c + di)`:

- Addition combines the real parts and imaginary parts.
- Multiplication uses `(ac - bd) + (ad + bc)i`.
- Division uses the conjugate-based formula implemented below.

`__repr__` formats positive and negative imaginary parts cleanly.

In [4]:
class ComplexNumber:
    def __init__(self, real, imag):
        self.real = real
        self.imag = imag

    def __add__(self, other):
        return ComplexNumber(self.real + other.real, self.imag + other.imag)

    def __sub__(self, other):
        return ComplexNumber(self.real - other.real, self.imag - other.imag)

    def __mul__(self, other):
        real_part = self.real * other.real - self.imag * other.imag
        imag_part = self.real * other.imag + self.imag * other.real
        return ComplexNumber(real_part, imag_part)

    def __truediv__(self, other):
        denominator = other.real**2 + other.imag**2
        real_part = (self.real * other.real + self.imag * other.imag) / denominator
        imag_part = (self.imag * other.real - self.real * other.imag) / denominator
        return ComplexNumber(real_part, imag_part)

    def __eq__(self, other):
        return self.real == other.real and self.imag == other.imag

    def __repr__(self):
        sign = "+" if self.imag >= 0 else "-"
        return f"{self.real} {sign} {abs(self.imag)}i"


c1 = ComplexNumber(2, 3)
c2 = ComplexNumber(1, 4)

print(c1 + c2)
print(c1 - c2)
print(c1 * c2)
print(c1 / c2)
print(c1 == c2)

3 + 7i
1 - 1i
-10 + 11i
0.8235294117647058 - 0.29411764705882354i
False


## 4. Comparing custom objects

A Product can decide that `<` and `==` compare prices. Python then lets us sort a list of products.

Comparison methods should first check the other object's type. `NotImplemented` tells Python: **I do not know how to compare myself with that type.**

In [5]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def __lt__(self, other):
        if not isinstance(other, Product):
            return NotImplemented
        return self.price < other.price

    def __eq__(self, other):
        if not isinstance(other, Product):
            return NotImplemented
        return self.price == other.price

    def __repr__(self):
        return f"Product({self.name!r}, {self.price})"


products = [Product("Bag", 800), Product("Pen", 20), Product("Book", 250)]
print("Sorted by price:", sorted(products))
print("Same price?", Product("Pen A", 20) == Product("Pen B", 20))

Sorted by price: [Product('Pen', 20), Product('Book', 250), Product('Bag', 800)]
Same price? True


## 5. Reflected operators: `__radd__` and `__rmul__`

Python first asks the object on the left to handle an operation. If it cannot, Python can ask the object on the right using a **reflected** method.

For `3 * vector`, the integer does not know how to multiply itself by our Vector. Python then tries `vector.__rmul__(3)`.

In [6]:
class SafeVector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __mul__(self, scalar):
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        return SafeVector(self.x * scalar, self.y * scalar)

    def __rmul__(self, scalar):
        return self * scalar

    def __repr__(self):
        return f"SafeVector({self.x}, {self.y})"


vector = SafeVector(2, 3)
print("Vector times number:", vector * 3)
print("Number times vector:", 3 * vector)

Vector times number: SafeVector(6, 9)
Number times vector: SafeVector(6, 9)


## 6. Unary operators

A unary operator works with one object. `-point` calls `point.__neg__()` and produces a point reflected across the origin.

In [7]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __neg__(self):
        return Point(-self.x, -self.y)

    def __repr__(self):
        return f"Point({self.x}, {self.y})"


point = Point(3, -4)
print("Original:", point)
print("Negative:", -point)

Original: Point(3, -4)
Negative: Point(-3, 4)


## 7. In-place operators

`counter += 3` tries `__iadd__`. An in-place method usually changes and returns the same object.

If `__iadd__` is missing, Python may fall back to `__add__` and assign the new result. Be deliberate about whether your class should mutate or create a new object.

In [8]:
class Score:
    def __init__(self, points=0):
        self.points = points

    def __iadd__(self, amount):
        self.points += amount
        return self

    def __repr__(self):
        return f"Score({self.points})"


score = Score(10)
original_id = id(score)
score += 5

print(score)
print("Same object?", id(score) == original_id)

Score(15)
Same object? True


## 8. Real-world example: a shopping basket

Here `+` combines two baskets into a new basket, and `len()` reports the number of items. Good operator overloading should feel natural and unsurprising.

In [9]:
class Basket:
    def __init__(self, items=None):
        self.items = list(items or [])

    def __add__(self, other):
        if not isinstance(other, Basket):
            return NotImplemented
        return Basket(self.items + other.items)

    def __len__(self):
        return len(self.items)

    def __repr__(self):
        return f"Basket({self.items!r})"


fruit_basket = Basket(["apple", "mango"])
snack_basket = Basket(["chips", "cookies"])
combined = fruit_basket + snack_basket

print(combined)
print("Total items:", len(combined))

Basket(['apple', 'mango', 'chips', 'cookies'])
Total items: 4


## Common mistakes and good rules

1. **Return a result.** An operator method that forgets `return` produces `None`.
2. **Return `NotImplemented` for unsupported types.** This gives Python a chance to try another valid route or raise a clear `TypeError`.
3. **Do not surprise the reader.** `+` should feel like adding or combining, not deleting data.
4. **Keep equality consistent.** If two objects compare equal and are hashable, they must produce the same hash.
5. **Avoid accidental mutation.** Normal `+` usually creates a new object; `+=` may mutate.
6. **Handle zero division.** A custom division method should reject a zero denominator clearly.
7. **Preserve useful types.** Adding two Vectors should usually return another Vector.

## Easy revision cheat sheet

| Syntax | Main method | Reflected/in-place version |
|---|---|---|
| `a + b` | `a.__add__(b)` | `b.__radd__(a)` / `a.__iadd__(b)` |
| `a - b` | `a.__sub__(b)` | `b.__rsub__(a)` / `a.__isub__(b)` |
| `a * b` | `a.__mul__(b)` | `b.__rmul__(a)` / `a.__imul__(b)` |
| `a / b` | `a.__truediv__(b)` | `b.__rtruediv__(a)` / `a.__itruediv__(b)` |
| `a == b` | `a.__eq__(b)` | No separate reflected name |
| `a < b` | `a.__lt__(b)` | Python may try `b.__gt__(a)` |
| `-a` | `a.__neg__()` | Unary operation |

### Five-second revision

**Operator overloading gives familiar operators a sensible meaning for your custom objects. It is implemented with magic methods.**